# Learning Objectives

* Reproduce the Caltech/IPAC *MeerKAT + 2MASS* Montage notebook (`MeerKAT.ipynb`) as a parallel Pegasus workflow.
* Generate and register input data, a container, and a workflow DAG through the Pegasus Python API.
* Visualize a workflow DAG before submitting it.
* Plan, submit, monitor, and pull statistics for a Pegasus workflow from a notebook.
* Inspect the 2MASS mosaics and the workflow's own validation against the notebook.

# MeerKAT / 2MASS Mosaic Workflow

The source notebook builds a three-color 2MASS (J, H, K) mosaic of the Galactic Center with the Montage Image
Mosaic Engine and blends a MeerKAT 1.28 GHz map over it. This workflow does the same work as a Pegasus DAG:
per band, the image table is split into chunks that are reprojected in parallel, then background-matched,
coadded and rendered; the composite, overlay and blend follow; a final job compares every product against
the notebook's. See `README.md`, `SPEC.md` and `TIER3_RUNBOOK.md` for the full description and the
validation tiers.

**Workflow Jobs (per band):**
1. `project_chunk` - `mProjExec` on one chunk of the band's images (one job per chunk)
2. `fitplanes` - `mOverlaps`/`mDiffExec`/`mFitExec`/`mBgModel`: background corrections
3. `mosaic` - `mBgExec` + `mAdd`: the corrected coadd (plus the uncorrected J coadd)
4. `render` - `mViewer`: PNG and thumbnail of each mosaic
5. `composite` / `overlay` / `blend` - three-color image and MeerKAT overlay (needs all three bands)
6. `validate` - Tier 1/2 comparison against the notebook's results

**Prerequisites:** [Pegasus WMS](https://pegasus.isi.edu/) >= 5.0, [HTCondor](https://htcondor.org/) >= 10.2,
Python 3.8+ with `pyyaml`, and [Apptainer](https://apptainer.org/) — all assumed to already be available in
this notebook's environment.

## 1. Prepare Sample Input Data

`bin/prefetch.py` (run inside the container) bootstraps the image tables for each band, downloads the 2MASS
images (~4 GB, ~25 min) and writes `data/manifest.json`, which is all the generator reads. With
`--mode prefetch` the local copies become the workflow's inputs. This is one-time: skip it if
`data/manifest.json` already exists.

In [ ]:
%%bash
if [ ! -f data/manifest.json ]; then
    apptainer exec Apptainer/Montage_Container.sif bin/prefetch.py --mode prefetch
else
    echo "Inputs already prepared: data/manifest.json"
fi

## 2. Build the Apptainer Container

The workflow's jobs run inside one Apptainer container (`Apptainer/Montage_Container.def`) that bundles
Montage and the Python wrappers' libraries. Building it can take a few minutes; skip the rebuild if the
`.sif` already exists.

In [ ]:
%%bash
if [ ! -f Apptainer/Montage_Container.sif ]; then
    apptainer build --ignore-fakeroot-command Apptainer/Montage_Container.sif Apptainer/Montage_Container.def
else
    echo "Container already built: Apptainer/Montage_Container.sif"
fi

## 3. Generate the Workflow

The following cell uses the `MeerKATMosaicWorkflow` class from `workflow_generator.py` — the same generator
used by the command-line tool — to build the Pegasus workflow DAG (site, transformation, and replica
catalogs) and writes it to `workflow.yml`. Its settings are parsed by the CLI's own parser
(`build_parser()`), so the notebook and the CLI share one set of options and defaults; nothing about the
workflow is defined here.

This is the quick development run: the J band only, first 20 images (`--subset 20 --bands J`). It still
coadds onto the full MeerKAT grid, so expect a mostly blank ~1.7 GB mosaic in a few minutes. For the full
run use `["--min-images", "460"]` (all three bands; see `TIER3_RUNBOOK.md` for disk needs).

`create_sites_catalog()` writes a self-contained site catalog for this machine's HTCondor pool (`local` +
`compute`). To run on a resource with a centrally hosted site catalog
([pegasus-site-catalogs](https://github.com/pegasushub/pegasus-site-catalogs/tree/main/conf), e.g.
`access-pegasus.yml`, `unity.yml`), set `HOSTED_SITE_CATALOG` instead — or set it once in `~/.pegasusrc`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from workflow_generator import MeerKATMosaicWorkflow, build_parser, normalise_bands

EXEC_SITE = "compute"
HOSTED_SITE_CATALOG = None  # e.g. "unity.yml"; None = local HTCondor catalog below

# The same options the CLI takes
args = build_parser().parse_args(["--subset", "20", "--bands", "J", "--output", "workflow.yml"])
args.bands = normalise_bands(args.bands)

workflow = MeerKATMosaicWorkflow(args)
workflow.load_manifest()
workflow.check_unique_filenames()
workflow.split_chunks()

print("Creating workflow properties...")
workflow.create_pegasus_properties(hosted_site_catalog=HOSTED_SITE_CATALOG)

if HOSTED_SITE_CATALOG is None:
    print("Creating execution sites...")
    workflow.create_sites_catalog(exec_site_name=EXEC_SITE)

print("Creating transformation catalog...")
workflow.create_transformation_catalog(exec_site_name=EXEC_SITE)

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating workflow DAG...")
workflow.create_workflow()

workflow.write()
workflow.summarise()
print("\nMeerKAT mosaic workflow has been generated!")

## 4. View the Generated Workflow DAG

Before submitting, visualize the workflow DAG using `pegasus-graphviz`. Each band fans out into one
`project_chunk` job per chunk of images, then funnels through `fitplanes`, `mosaic` and `render`; every
product feeds the final `validate` job.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image

Image(filename="workflow.png")

## 5. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the `compute` site defined in the site catalog
chosen above, with `cleanup=inplace` so the ~9 GB of intermediate chunk archives are removed as soon as
nothing needs them.

The CLI version of this generator (`workflow_generator.py`) writes the same workflow non-interactively and
prints the `pegasus-plan` command; it never submits by itself. Submitting is this explicit step.

In [ ]:
workflow.plan_submit(exec_site_name=EXEC_SITE)

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed
states. `wait()` blocks (with a progress bar) until the workflow completes or fails.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 6. Statistics

Once the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 7. Examine the Results

The workflow stages the following final outputs to `output/`:

| Output | Description |
|--------|-------------|
| `2mass{j,h,k}.fits`, `.png`, `_small.png` | Background-corrected mosaic per band, rendered |
| `uncorrected.fits`, `.png` | The notebook's illustrative uncorrected J coadd |
| `2MASS.png`, `MeerKAT.png` (+ `*Stretch.png`, `*_small.png`) | Three-color composite, MeerKAT overlay and blend (all three bands only) |
| `*.tbl`, `*.status.json` | Montage tables and per-step status records |
| `validation_report.md`, `.json` | Comparison against the notebook's products |

In [ ]:
!ls -lt output/

The J-band mosaic from the 20-image subset — only part of the grid is covered, as expected:

In [ ]:
Image(filename="output/2massj_small.png")

And the workflow's own comparison against the notebook (checks the subset cannot meet are relaxed or skipped):

In [ ]:
!cat output/validation_report.md